In [34]:
import sys
!{sys.executable} -m pip install pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


In [35]:
import numpy as np

In [36]:
import pandas as pd

results = pd.read_csv('../data/raw/results.csv')
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [39]:
results.dtypes

date              str
home_team         str
away_team         str
home_score    float64
away_score    float64
tournament        str
city              str
country           str
neutral          bool
dtype: object

In [40]:
results["date"].min()

'1872-11-30'

In [41]:
results["date"].max()

'2026-06-27'

In [42]:
results['date']=pd.to_datetime(results['date'])
results.dtypes

date          datetime64[us]
home_team                str
away_team                str
home_score           float64
away_score           float64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object

In [43]:
results = results.dropna(subset=['home_score', 'away_score'])
results.shape

(49215, 9)

In [44]:
results = results.sort_values(by='date', ascending=True)
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [45]:
former_names = pd.read_csv('../data/raw/former_names.csv')
former_names.shape

(36, 4)

In [46]:
name_map = dict(zip(former_names['former'], former_names['current']))
results['home_team'] = results['home_team'].replace(name_map)
results['away_team'] = results['away_team'].replace(name_map)

In [47]:
results[results['away_team'] == 'Dahomey']

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral


In [48]:
conditions = [results['home_score'] > results['away_score'] , results['home_score'] < results['away_score']]
choices = ['HOME_WIN', 'AWAY_WIN']
results['result'] = np.select(conditions, choices, default='DRAW')

In [49]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,DRAW
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,HOME_WIN
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,HOME_WIN
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,DRAW
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,HOME_WIN


In [55]:
results[results['tournament'].str.contains('Nations League', case=False)]['tournament'].unique()

<StringArray>
['CONCACAF Nations League qualification',
                   'UEFA Nations League',
               'CONCACAF Nations League']
Length: 3, dtype: str

In [68]:
conditions = [results['tournament'].str.contains('World Cup') & ~results['tournament'].str.contains('qualification'),
              results['tournament'].str.contains('qualification') | results['tournament'].str.contains('qualifier'), 
              results['tournament'].str.contains('Olympic Games', case=False),
              (results['tournament'].str.contains('Copa América', case=False) |
                results['tournament'].str.contains('UEFA Euro', case=False) |
                results['tournament'].str.contains('African Cup of Nations', case=False) |
                results['tournament'].str.contains('AFC Asian Cup', case=False) |
                results['tournament'].str.contains('Gold Cup', case=False)) & ~results['tournament'].str.contains('qualification', case=False),
              results['tournament'].str.contains('Nations League', case=False) & ~results['tournament'].str.contains('qualification', case=False),
              results['tournament'] == 'Friendly']
choices = ['WORLD_CUP', 'QUALIFICATION', 'OLYMPICS', 'CONTINENTAL_CUP', 'NATIONS_LEAGUE', 'FRIENDLY']
results['match_importance'] = np.select(conditions, choices, default='OTHER')

In [69]:
results['match_importance'].value_counts()

match_importance
FRIENDLY           18252
QUALIFICATION      15927
OTHER               9911
CONTINENTAL_CUP     2943
NATIONS_LEAGUE      1080
WORLD_CUP           1024
OLYMPICS              78
Name: count, dtype: int64

In [70]:
shootouts = pd.read_csv('../data/raw/shootouts.csv')
shootouts.head()

,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN
4,1972-05-19,Thailand,Cambodia,Thailand,NaN


In [73]:
shootouts = shootouts.drop(columns=['first_shooter'])

KeyError: "['first_shooter'] not found in axis"

In [78]:
shootouts['date'] = pd.to_datetime(shootouts['date'])

In [79]:
merged_results = pd.merge(results, shootouts, on=['date', 'home_team', 'away_team'], how='left')

In [81]:
merged_results['winner'].value_counts()

winner
South Korea                     15
Argentina                       15
Egypt                           15
Zambia                          14
Thailand                        13
                                ..
United States Virgin Islands     1
Vanuatu                          1
Palestine                        1
Comoros                          1
Azerbaijan                       1
Name: count, Length: 181, dtype: int64

In [82]:
merged_results.shape

(49215, 12)

In [83]:
merged_results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,match_importance,winner
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,DRAW,FRIENDLY,NaN
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,HOME_WIN,FRIENDLY,NaN
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,HOME_WIN,FRIENDLY,NaN
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,DRAW,FRIENDLY,NaN
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,HOME_WIN,FRIENDLY,NaN


In [84]:
merged_results.to_csv('../data/processed/matches_clean.csv', index=False)

In [85]:
pd.read_csv('../data/processed/matches_clean.csv').shape

(49215, 12)